In [1]:
from IPython.display import Image, display
import numpy as np
import reddemcee
np.random.seed(1234)

# Parallelization and Vectorization
You can parallelize the sampler by using a pool, multiprocessing is recommended, but multiprocess and schwimmbad were tested as well.

In MacOS systems, you might need to add an additional safeguard.

In [2]:
import multiprocessing as mp

# MacOS safeguard
import sys
is_darwin = False
if sys.platform == 'darwin':
    is_darwin = True

if is_darwin:
    try:
        mp.set_start_method('fork')  # not spawn
    except:
        print('\n MacOS safeguard \n multiprocessing.set_start_method(fork, force=True)\n')
        print(' Method set to:', mp.get_start_method())

You can check how many threads you have available by using:

In [3]:
mp.cpu_count()

15

We will build a likelihood that holds each thread for a set amount of time:

In [4]:
import time

def loglike(theta):
    t = time.time() + np.random.uniform(0.005, 0.008)
    while True:
        if time.time() >= t:
            break
    return -0.5 * np.sum(theta**2)

def logprior(theta):
    return 0.

## Serial
This likelihood function will sleep for a random second fraction when called. We start by evaluating the performance in a serial initialization of the sampler:

In [5]:
ndim_ = 2

setup = [2, 20, 40, 2]
ntemps, nwalkers, nsweeps, nsteps = setup

p0 = list(np.random.randn(ntemps, nwalkers, ndim_))

In [6]:
sampler_s = reddemcee.PTSampler(nwalkers, ndim_,
                              loglike, logprior,
                              ntemps=ntemps,
                              )

start = time.time()
samp_s = sampler_s.run_mcmc(p0, nsweeps, nsteps)
time_serial = time.time() - start

print(f'Serial took {time_serial:.1f} seconds')

Serial took 21.5 seconds


## Parallel

The HD 20794 analysis used 24 threads, as reported in the [reddemcee paper](https://ui.adsabs.harvard.edu/abs/2026A%26A...706A.323P).

In MacOS, you might need a safeguard:

In [7]:
with mp.Pool(10) as mypool:
    sampler_p = reddemcee.PTSampler(nwalkers, ndim_,
                                  loglike, logprior,
                                  ntemps=ntemps,
                                  pool=mypool)
    start = time.time()
    samp_p = sampler_p.run_mcmc(p0, nsweeps, nsteps)
    time_parallel = time.time() - start

print(f'Parallel took {time_parallel:.1f} seconds')

Parallel took 2.9 seconds


In [13]:
print(f'{"Example":<22} {"Mode":<12} {"Time (s)":>8}')
for example, mode, elapsed in [
    ('Gaussian delayed', 'Serial', time_serial),
    ('Gaussian delayed', 'Pool', time_parallel),
]:
    print(f'{example:<22} {mode:<12} {elapsed:8.3f}')

Example                Mode         Time (s)
Gaussian delayed       Serial         21.470
Gaussian delayed       Pool            2.887


That's fast!

## Vectorization

With `vectorize=True`, the sampler passes an array of shape `(npoints, ndim)` to logl and logp. Return one value per point, with shape `(npoints,)`. A constant prior can also return a single number, as in the first example.

Use `axis=-1` to reduce over the parameters of each point. The prior returns `-np.inf` outside the box, and the sampler skips the likelihood for those.
The batch size can vary during sampling!

In [ ]:
ndim_vec = 3
p0_vec = np.random.randn(ntemps, nwalkers, ndim_vec)

def loglike_vec(theta):
    return -0.5 * np.sum(theta**2, axis=-1)

def logprior_vec(theta):
    inside = np.all(np.abs(theta) < 10, axis=-1)
    return np.where(inside, -theta.shape[-1] * np.log(20.), -np.inf)

theta = np.array([[0., 0., 0.], [1., -1., 2.], [11., 0., 0.]])

The same functions also work on one point at a time. We first time a serial run with this 3D Gaussian likelihood and box prior.

In [ ]:
sampler_scalar = reddemcee.PTSampler(nwalkers, ndim_vec,
                                    loglike_vec, logprior_vec,
                                    ntemps=ntemps)

start = time.perf_counter()
sampler_scalar.run_mcmc(p0_vec, nsweeps, nsteps)
time_scalar = time.perf_counter() - start

Now we use the same starting points with `vectorize=True`. In this mode, probability evaluation bypasses the pool.

In [ ]:
sampler_v = reddemcee.PTSampler(nwalkers, ndim_vec,
                              loglike_vec, logprior_vec,
                              ntemps=ntemps,
                              vectorize=True)

start = time.perf_counter()
sampler_v.run_mcmc(p0_vec, nsweeps, nsteps)
time_vectorized = time.perf_counter() - start

Compare timings within each example.

In [ ]:
print(f'{"Example":<22} {"Mode":<12} {"Time (s)":>8}')
for example, mode, elapsed in [
    (f'{ndim_vec}D Gaussian', 'Serial', time_scalar),
    (f'{ndim_vec}D Gaussian', 'Vectorized', time_vectorized),
]:
    print(f'{example:<22} {mode:<12} {elapsed:8.3f}')